# Automated EDA Profiling — (Tool Comparison)

Before diving into manual EDA, it's worth running the data through automated 
profilers first — they surface structure, quality issues, and relationships 
in seconds instead of writing the same boilerplate `.describe()` / `.isna().sum()` 
/ `groupby()` code every time.

**Important:** these tools don't replace real EDA — they replace the *boring 
first pass* of it. Each one is built for a different job, not a "better vs 
worse" ranking:

| Tool | Best for | Mode |
|---|---|---|
| **ydata-profiling** | Fast structural audit of a new dataset — types, missing %, correlations | Static report, passive |
| **d-tale** | Actively testing hypotheses — filter, drill into a column, check a scatter | Live interactive app |
| **Sweetviz** | Relating every feature to a target, or comparing train vs test for drift | Static report, target-aware |

### Suggested workflow
1. **First contact with data** → `ydata-profiling` — sanity check shape, types, missing values, obvious junk columns
2. **Once got a target** → `Sweetviz` (with `target_feat` set) — see which features actually move the target
3. **Anything Sweetviz flags as interesting** → `d-tale` — drill in live, filter, check the actual scatter/rows behind the number
4. **Right before training** → `Sweetviz.compare()` on train/test — catch distribution drift before it becomes a silent bug

Each tool's section below has its own reference notes on usage, strengths, 
and limitations.

In [1]:
# Importing the dataset
import pandas as pd 
df = pd.read_csv("../data/Titanic.csv")

# Overview of dataset
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 1. ydata-profiling 

**What it is:** Automated data-audit tool. Generates a static HTML report 
summarizing structure, quality, and relationships in a dataframe — a fast 
first-pass "what am I dealing with?" tool, NOT a substitute for real EDA.


### Report Sections

| Section | What it shows | Why it matters |
|---|---|---|
| **Overview** | Row/col counts, memory size, % missing cells, duplicate rows | Sanity check — catches dupes/shape mismatches early |
| **Variables** | Per column: inferred type, distinct count, missing %; numeric → mean/std/quantiles/skew/zeros; categorical → value counts, cardinality | Spot wrong dtypes, high-cardinality IDs, constant columns |
| **Interactions/Correlations** | Pearson, Spearman, Cramér's V, etc. (pairwise) | Flags relationships without manual plotting — biggest time-saver |
| **Missing values** | Matrix/heatmap of null locations | Reveals if missingness is random or clustered (structural) |
| **Sample** | Head/tail rows | Gut check on raw data |

### Strengths
- Instant overview in first minutes with a new dataset
- Good at catching data quality issues before modeling
- Correlation matrix is genuinely useful to skim early

### Limitations (important)
- Has **no concept of target variable** — treats all correlations as equally interesting
- Can't explain *why* something is missing (e.g. missing-at-random vs. structural, like `Cabin` missingness likely tied to `Pclass`)
- Descriptive only — it summarizes, it doesn't reason about your problem

### Practice habit
Before reading the correlation section, **predict** which 2-3 features will 
correlate most with target and why — then check if you were right. 
Turns a passive report into an active exercise.

In [4]:
# Importing ydata profiler
from ydata_profiling import ProfileReport

prof = ProfileReport(df)
prof.to_file(output_file='../data/genrated_data/ydata_profile_titanic.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:00<00:00, 103563.06it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# 2. d-tale

**What it is:** Interactive live web app for exploring a dataframe — 
unlike ydata's static report, d-tale runs a local server backed by your 
actual df, so you interrogate the data in real time instead of reading 
a fixed summary.



### Core Features

| Feature | What it does | Why it matters |
|---|---|---|
| **Grid view** | Sortable, live-filterable spreadsheet view | Filter with pandas-syntax queries (e.g. `Age > 30 and Pclass == 1`) and see row counts update instantly |
| **Column analysis** | Click a column → histogram (numeric) or bar chart (categorical) + describe() stats, on demand | Spot skew, outliers, suspicious spikes (e.g. round-number imputed ages) one column at a time |
| **Correlations** | Interactive heatmap — click a cell to see the scatter plot for that pair | Turns a static correlation number into a visual you can sanity-check immediately |
| **Charts** | Drag-and-drop line/bar/scatter/heatmap builder | Build visuals without writing matplotlib/seaborn |
| **Code export** | Every UI action (filter, chart, transform) shows equivalent pandas code | Explore visually, then copy working code into your script |
| **Custom columns/filters** | Formula bar to create new columns in-UI | Quick hypothesis testing without leaving the browser |

### Strengths
- Active exploration, not passive reading — forces you to form and test hypotheses
- Code export bridges "playing around" and "real script"
- On-demand column analysis avoids scrolling past irrelevant stats

### Limitations
- Local dev tool — session tied to your kernel, not easily shareable like ydata's HTML
- No auto-generated "insight narrative" — it's a workbench, not a report
- Runs a local server — minor setup/security consideration on shared machines

### Practice habit
Before applying any filter or opening a column's analysis, **predict** the 
result (row count, skew direction, missing %) — then check. Keeps it from 
becoming passive clicking.

In [5]:
# Import d-tale
import dtale
dtale.show(df,enable_custom_filters = True) # Change the flag as 'False' if custom filter is not required 

2026-07-28 23:46:33,012 - WARNING  - Custom filtering enabled. Custom filters are vulnerable to code injection attacks, please only use in trusted environments.


# 3. Sweetviz 

**What it is:** Static HTML report generator, like ydata, but built around 
ONE differentiator: relating every feature to a target column (and/or 
comparing two datasets, e.g. train vs test).


### Core Features

| Feature | What it does | Why it matters |
|---|---|---|
| **Target analysis** | Every feature's distribution gets an overlaid target-rate line/bars | Instantly reveals which features actually predict the target — e.g. Sex/Pclass vs Survived |
| **Train/test comparison** | Overlays two datasets' distributions per column | Catches data drift/leakage before model training — no other tool here does this |
| **Associations** | Correlation matrix, split into Numerical (Pearson) vs Categorical (correlation ratio) | Cleaner than ydata's mixed correlation view |
| **Single-page layout** | All stats in one scroll, no tabs | Fast to skim, but dense |

### Strengths
- Only tool of the three that's target-aware out of the box
- Train/test drift detection is genuinely unique among these three
- Categorical vs numerical association split is clearer than ydata's

### Limitations
- No interactivity (can't click into a scatter like d-tale)
- No code export
- Without a target/comparison set, it's just a weaker ydata — don't run it plain
- Report generation slows down on wide dataframes

### Practice habit
Never run `sv.analyze(df)` alone — always pass `target_feat`. If you don't 
have an obvious target, ask "what am I even trying to learn?" before 
reaching for Sweetviz specifically — it's not a general-purpose tool.

In [6]:
import sweetviz as sv

# Basic (no target) — weak mode
sweet_repo = sv.analyze(df)
sweet_repo.show_html("../data/genrated_data/sweetviz_report_titanic.html")

                                             |          | [  0%]   00:00 -> (? left)

Report ../data/genrated_data/sweetviz_report_titanic.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [7]:
# Target-aware (the useful mode)

sweet_repo = sv.analyze(df, target_feat="Survived")
sweet_repo.show_html("../data/genrated_data/sweetviz_target_titanic.html")

                                             |          | [  0%]   00:00 -> (? left)

Report ../data/genrated_data/sweetviz_target_titanic.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [ ]:
# Train/test comparison (killer feature)
#NOTE: To Run this command split the data into train/test 
report = sv.compare([train_df, "Train"], [test_df, "Test"], target_feat="Survived")
report.show_html("../data/genrated_data/sweetviz_compare_titanic.html")

In [9]:
# Analyse Survival chance with Sex

report = sv.compare_intra(df, df["Sex"] == "male", ["Male", "Female"], target_feat="Survived")
report.show_html("../data/genrated_data/sweetviz_compare_with_sex_titanic.html")

                                             |          | [  0%]   00:00 -> (? left)

Report ../data/genrated_data/sweetviz_compare_with_sex_titanic.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [10]:
# Analyse just relevant feature 
repo=sv.analyze(df, pairwise_analysis="on",target_feat="Survived", feat_cfg=sv.FeatureConfig(skip=["PassengerId", "Name", "Ticket"])) #skip the irrelevant features

# Try these as well:
#1. sv.FeatureConfig(force_cat=["Pclass"])
#2. sv.analyze(df, pairwise_analysis="on")

repo.show_html("../data/genrated_data/sweetviz_drop_analy_titanic.html")

                                             |          | [  0%]   00:00 -> (? left)

Report ../data/genrated_data/sweetviz_drop_analy_titanic.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


## ✅ Final Takeaways — Auto-EDA Profilers

**Big lesson:** no single tool "wins" — each optimizes for a different stage 
of exploration. Judging Sweetviz by its no-target output (like I first did) 
misses its actual purpose entirely — always check *how* a tool is meant to 
be used before writing it off.

### Quick decision guide
- **"I just loaded this dataset, what am I working with?"** → ydata-profiling
- **"I have a hypothesis, let me check it right now"** → d-tale
- **"Which features actually matter for my target?"** → Sweetviz (`target_feat` set)
- **"Will my model see different data in production than in training?"** → Sweetviz `.compare()`

### Core habit to carry forward (not tool-specific)
Before opening any report/chart — **predict the answer first** (a %, a 
correlation direction, a skew). Then check. This is what turns "reading an 
auto-EDA report" into actually doing EDA — the tools only save you the 
manual computation, not the thinking.

### Practical notes learned this session
- Sweetviz without `target_feat` is just a weaker ydata — always set a target
- Skip ID-like columns (`PassengerId`, `Name`, `Ticket`) in Sweetviz via 
  `FeatureConfig(skip=[...])` for a cleaner, faster report
- d-tale's code-export feature is the bridge between "poking around" and 
  a real script — use it instead of retyping filters manually
- ydata's correlation matrix is a good first flag, but it has no idea 
  what your target is — don't stop analysis there

